### Transportation Optimisation:

AI chat used, output before any debugging and optimising of code (https://chatgpt.com/share/69c8c6f4-2e9c-839e-9d53-158b85461125)

### Explanation of the Code Cell

This code cell focuses on **importing required libraries**, **loading multiple datasets**, and **fixing text encoding issues in the data**.

---

### 1. Data Loading

Three datasets are loaded from extracted CSV files (https://www.kaggle.com/datasets/cdthinh/japanese-train-system/data):

- `join.csv`: Likely contains relationships or connections between entities  
- `line.csv`: Railway or transport line data  
- `station.csv`: Station-level data including locations and attributes  

All files are read using `latin1` encoding to prevent character decoding errors.

---

### 3. Text Encoding Fix Function

#### `fix_mojibake(x)`
This function attempts to correct **mojibake (garbled text caused by encoding mismatches)**:

- If the input is a string:
  - It re-encodes it using `latin1` and decodes it back into `utf-8`
- If conversion fails, it safely returns the original value

---

### 4. DataFrame-Wide Cleaning Function

#### `fix_dataframe(df)`
This function applies the encoding fix across an entire DataFrame:

- Iterates through all columns with object (string) data types
- Applies `fix_mojibake` to every cell in those columns

---

### 5. Cleaned DataFrames

The final step creates cleaned versions of the datasets:

- `df_joins`
- `df_lines`
- `df_stations`

These cleaned DataFrames ensure that all text fields are properly encoded and readable for downstream analysis and modeling.

In [2]:
import pandas as pd
import numpy as np
import collections
from gurobipy import Model, GRB, quicksum
import folium
from folium import PolyLine
from folium.plugins import MarkerCluster
from folium.plugins import PolyLineTextPath
import requests
from IPython.display import display

joins = pd.read_csv("join20240426.csv")
lines = pd.read_csv("line20240426.csv")
stations = pd.read_csv("station20240426.csv")

def fix_mojibake(x):
    try:
        return x.encode('latin1').decode('utf-8') if isinstance(x, str) else x
    except:
        return x

def fix_dataframe(df):
    for col in df.select_dtypes(include="object"):
        df[col] = df[col].map(fix_mojibake)
    return df

df_joins = fix_dataframe(joins)
df_lines = fix_dataframe(lines)
df_stations = fix_dataframe(stations)

In [3]:
df_lines.head()

,line_cd,company_cd,line_name,line_name_h,lon,lat,e_status,e_sort
0,1001,3,中央新幹線,中央新幹線,137.493896,35.411438,1,1001
1,1002,3,東海道新幹線,東海道新幹線,137.721489,35.144122,0,1002
2,1003,4,山陽新幹線,山陽新幹線,133.147896,34.419338,0,1003
3,1004,2,東北新幹線,東北新幹線,140.763192,38.274267,0,1004
4,1005,2,上越新幹線,上越新幹線,139.121488,36.798565,0,1005


In [4]:
df_lines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 622 entries, 0 to 621
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   line_cd      622 non-null    int64  
 1   company_cd   622 non-null    int64  
 2   line_name    622 non-null    object 
 3   line_name_h  622 non-null    object 
 4   lon          622 non-null    float64
 5   lat          622 non-null    float64
 6   e_status     622 non-null    int64  
 7   e_sort       622 non-null    int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 39.0+ KB


### Explanation of the Code Cell

This code cell defines a **data preprocessing function** that standardises data types, prepares multiple datasets for analysis, and ensures consistency across a multi-layer transport + location dataset.

---

## 1. Dataset Overview

The function operates on **5 related DataFrames**, each representing a different component of the system:

---

### (1) `df_lines` — Train Line Information

**Columns:**
`line_cd, company_cd, line_name, line_name_h, lon, lat, e_status, e_sort`

**Meaning:**

- `line_cd`: Line identifier (used to match station and join data)
- `line_name`: Line name (language 1)
- `line_name_h`: Line name (language 2)
- `lon`, `lat`: Geographic reference (currently not used but retained)
- `e_status`, `e_sort`: Extra metadata (not used currently)

---

### (2) `df_stations` — Station Information

**Columns:**
`station_cd, station_g_cd, station_name, line_cd, post, address, lon, lat, open_ymd, close_ymd, e_status, e_sort`

**Meaning:**

- `station_cd`: Unique station ID (7 digits; structured encoding of line + station)
- `station_g_cd`: Grouped station ID (same logical identity as station_cd grouping)
- `station_name`: Station name
- `line_cd`: Associated train line
- `post`: Postal code
- `address`: Station address
- `lon`, `lat`: Geographic coordinates (critical for mapping)
- `close_ymd`: Closing date (YYYY-MM-DD format)
- `open_ymd`, `e_status`, `e_sort`: Metadata (not used currently)

---

### (3) `df_joins` — Station Connectivity (Graph Edges)

**Columns:**
`line_cd, station_cd1, station_cd2`

**Meaning:**

- `line_cd`: Line identifier linking to `df_lines`
- `station_cd1`: Origin station in a connection
- `station_cd2`: Destination station in a connection

**Key structure:**

- Encodes the **rail network graph structure**
- Stations are connected as edges between nodes
- A station may appear multiple times if it connects to multiple stations
- Station codes encode both **line ID + station position within line**

---

### (4) `df_locations` — External Attractions Dataset

**Source:**
`selected_attractions.csv`

**Columns:**
`title, review_score, city, all_categories, latitude, longitude`

**Meaning:**

- `title`: Attraction name
- `review_score`, `city`, `all_categories`: Metadata (not used currently)
- `latitude`, `longitude`: Geographic coordinates (used for mapping and routing)

---

## 2. Function Overview

### `preprocess_data(df_lines, df_stations, df_joins, df_companies, df_locations)`

This function standardises and prepares all datasets for downstream graph modelling, optimisation, and geospatial analysis.

---

## 3. Data Type Standardisation

To ensure consistency across joins, graph construction, and modelling:

### 🚆 `df_lines`
- `line_cd` → string  
- `company_cd` → string  

### 🚉 `df_stations`
- `station_cd` → string  
- `station_g_cd` → string  
- `line_cd` → string  
- `lat` → float  
- `lon` → float  

### 🔗 `df_joins`
- `line_cd` → string  
- `station_cd1` → string  
- `station_cd2` → string  

### 🏢 `df_companies`
- `company_cd` → string  

### 📍 `df_locations`
- `latitude` → float  
- `longitude` → float  

---

## 4. Location Node Construction

- The index of `df_locations` is reset to ensure sequential ordering
- A new column `node_id` is created based on this index

This enables:
- Graph node identification
- Route optimisation modelling (e.g., Gurobi)
- Mapping and indexing consistency

---

## 5. Output

The function returns cleaned and standardised versions of:

- `df_lines`
- `df_stations`
- `df_joins`
- `df_locations`

In [5]:
def preprocess_data(df_lines, df_stations, df_joins, df_locations):

    print("🔧 Preprocessing data...")

    df_lines["line_cd"] = df_lines["line_cd"].astype(str)
    df_lines["company_cd"] = df_lines["company_cd"].astype(str)

    df_stations["station_cd"] = df_stations["station_cd"].astype(str)
    df_stations["station_g_cd"] = df_stations["station_g_cd"].astype(str)
    df_stations["line_cd"] = df_stations["line_cd"].astype(str)
    df_stations["lat"] = df_stations["lat"].astype(float)
    df_stations["lon"] = df_stations["lon"].astype(float)

    df_joins["line_cd"] = df_joins["line_cd"].astype(str)
    df_joins["station_cd1"] = df_joins["station_cd1"].astype(str)
    df_joins["station_cd2"] = df_joins["station_cd2"].astype(str)

    df_locations["latitude"] = df_locations["latitude"].astype(float)
    df_locations["longitude"] = df_locations["longitude"].astype(float)

    df_locations = df_locations.reset_index(drop=True)
    df_locations["node_id"] = df_locations.index

    print("✅ Data cleaned")

    return df_lines, df_stations, df_joins, df_locations

### Explanation of the Code Cell

This code cell defines a function to compute the **real-world distance between two geographic coordinates** using the **Haversine formula**.

---

## 1. Function Overview

### `actual_distance(coord1, coord2)`

This function calculates the shortest distance over the Earth’s surface between two points, given their latitude and longitude.

- Input:
  - `coord1 = (lat1, lon1)`
  - `coord2 = (lat2, lon2)`
- Output:
  - Distance in **kilometres (km)**

---

## 2. Estimated Earth Radius Constant

```python
R = 6371

In [6]:
def actual_distance(coord1, coord2):
    R = 6371

    lat1, lon1 = np.radians(coord1)
    lat2, lon2 = np.radians(coord2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return R * c

### Explanation of the Code Cell

This code cell defines a function that **clusters location nodes based on their distance from a starting point**, grouping nearby locations together into fixed-size clusters.

---

## 1. Function Overview

### `cluster_locations(df_locations, start_lat, start_lon, K=3)`

The function:
- Computes the distance of each location from a given starting point
- Sorts locations by proximity
- Groups them into clusters of size **K (default = 3)**

---

## 2. Distance Calculation

Each location’s distance from the starting coordinate is computed using the **Haversine distance function (`actual_distance`)**:


In [7]:
def cluster_locations(df_locations, start_lat, start_lon, K=3):

    print("\n🔹 Clustering locations...")

    df = df_locations.copy()

    # ---- distance from start ----
    df["dist_start"] = df.apply(
        lambda r: actual_distance(
            (start_lat, start_lon),
            (r["latitude"], r["longitude"])
        ),
        axis=1
    )

    df = df.sort_values("dist_start")

    clusters = []
    current = []

    for _, row in df.iterrows():

        current.append({
            "id": row["node_id"],
            "name": row["title"],
            "lat": row["latitude"],
            "lon": row["longitude"],
            "distance_from_start": row["dist_start"]   # ✅ added here
        })

        if len(current) == K:
            clusters.append(pd.DataFrame(current))
            current = []

    if current:
        clusters.append(pd.DataFrame(current))

    print(f"✅ Created {len(clusters)} clusters")

    return clusters

In [8]:
def greedy_spatial_cluster(df_locations, K=3):

    df = df_locations.copy().reset_index(drop=True)

    remaining = df.copy()

    clusters = []

    while len(remaining) > 0:

        # pick first point
        base = remaining.iloc[0]
        base_coord = (base["latitude"], base["longitude"])

        # compute distances to all others
        remaining["dist"] = remaining.apply(
            lambda r: actual_distance(
                base_coord,
                (r["latitude"], r["longitude"])
            ),
            axis=1
        )

        # pick K closest points
        cluster = remaining.nsmallest(K, "dist")

        clusters.append(cluster.drop(columns="dist"))

        # remove them from remaining
        remaining = remaining.drop(cluster.index).reset_index(drop=True)
        
        for cluster in clusters:
            cluster.rename(columns={"node_id": "id", "latitude": "lat", "longitude": "lon", "title": "name"}, inplace=True)

    return clusters

In [9]:
import numpy as np
import pandas as pd

def balanced_geo_cluster(df_locations, K=3):
    """
    Cluster attractions into groups of size K using angular grouping
    around the global centroid.
    Returns list of DataFrames with columns: id, name, lat, lon
    """

    df = df_locations.copy().reset_index(drop=True)

    # standardise column names if needed
    if "node_id" in df.columns and "id" not in df.columns:
        df = df.rename(columns={"node_id": "id"})
    if "title" in df.columns and "name" not in df.columns:
        df = df.rename(columns={"title": "name"})
    if "latitude" in df.columns and "lat" not in df.columns:
        df = df.rename(columns={"latitude": "lat"})
    if "longitude" in df.columns and "lon" not in df.columns:
        df = df.rename(columns={"longitude": "lon"})

    # global centroid
    center_lat = df["lat"].mean()
    center_lon = df["lon"].mean()

    # angle from centroid
    df["angle"] = df.apply(
        lambda r: np.arctan2(r["lat"] - center_lat, r["lon"] - center_lon),
        axis=1
    )

    # sort around the center
    df = df.sort_values("angle").reset_index(drop=True)

    clusters = []
    for i in range(0, len(df), K):
        cluster_df = df.iloc[i:i+K][["id", "name", "lat", "lon"]].copy()
        clusters.append(cluster_df)

    return clusters

In [10]:
def cluster_cost(cluster_df):
    coords = cluster_df[["lat", "lon"]].values.tolist()
    total = 0.0
    for i in range(len(coords)):
        for j in range(i + 1, len(coords)):
            total += actual_distance(coords[i], coords[j])
    return total

def total_cluster_cost(clusters):
    return sum(cluster_cost(c) for c in clusters)

def refine_clusters_by_swapping(clusters, max_iter=20):
    """
    Improve clusters by swapping points between clusters
    if total within-cluster distance decreases.
    """

    clusters = [c.reset_index(drop=True).copy() for c in clusters]

    improved = True
    iteration = 0

    while improved and iteration < max_iter:
        improved = False
        iteration += 1

        current_best = total_cluster_cost(clusters)

        for a in range(len(clusters)):
            for b in range(a + 1, len(clusters)):
                ca = clusters[a]
                cb = clusters[b]

                for i in range(len(ca)):
                    for j in range(len(cb)):
                        new_ca = ca.copy()
                        new_cb = cb.copy()

                        row_a = ca.iloc[i].copy()
                        row_b = cb.iloc[j].copy()

                        new_ca.iloc[i] = row_b
                        new_cb.iloc[j] = row_a

                        new_clusters = clusters.copy()
                        new_clusters[a] = new_ca
                        new_clusters[b] = new_cb

                        new_cost = total_cluster_cost(new_clusters)

                        if new_cost < current_best:
                            clusters = [c.copy() for c in new_clusters]
                            current_best = new_cost
                            improved = True

        # continue until no improving swap exists

    return clusters

### Explanation of the Code Cell

This code cell defines a function that constructs a **line-aware railway station graph**, where nodes represent stations on specific lines and edges represent either **movement along a line** or **transfers between lines**.

---

## 1. Function Overview

### `build_station_graph(df_stations, df_joins)`

This function builds a **graph data structure** representing the railway network:

- Nodes → `(station_g_cd, line_cd)`  
- Edges → connections between nodes (same line or transfer)

It returns a graph suitable for:
- Pathfinding (e.g., shortest path)
- Route optimisation
- Network traversal


In [11]:
def build_station_graph(df_stations, df_joins):

    print("\n🔹 Building LINE-AWARE station graph...")

    graph = collections.defaultdict(list)

    # -------------------------
    # Create node: (station_g_cd, line_cd)
    # -------------------------
    station_nodes = df_stations[["station_g_cd", "line_cd"]].drop_duplicates()

    # -------------------------
    # 1) SAME LINE MOVEMENT
    # -------------------------
    df_sorted = df_stations.sort_values(["line_cd", "station_cd"])

    for line, group in df_sorted.groupby("line_cd"):

        stations = group["station_g_cd"].tolist()

        for i in range(len(stations) - 1):
            a = (stations[i], line)
            b = (stations[i+1], line)

            graph[a].append(b)
            graph[b].append(a)

    # -------------------------
    # 2) TRANSFERS (same station, different lines)
    # -------------------------
    for station, group in df_stations.groupby("station_g_cd"):

        lines = group["line_cd"].unique()

        for i in range(len(lines)):
            for j in range(i+1, len(lines)):

                a = (station, lines[i])
                b = (station, lines[j])

                graph[a].append(b)
                graph[b].append(a)

    return graph

### Explanation of the Code Cell

This code cell defines a function that finds the **nearest train station to a given geographic coordinate** based on real-world distance.

---

## 1. Function Overview

### `nearest_station(coord, df_stations)`

The function:
- Takes a coordinate (latitude, longitude)
- Compares it against all station locations
- Returns the **closest station** and its **distance**


In [12]:
def nearest_station(coord, df_stations):
    best_dist = float("inf")
    best_station = None

    for _, row in df_stations.iterrows():
        st_coord = (row["lat"], row["lon"])
        d = actual_distance(coord, st_coord)

        if d < best_dist:
            best_dist = d
            best_station = row

    return best_station, best_dist

7) train_route() function, based on guidance provided in (https://www.kaggle.com/datasets/cdthinh/japanese-train-system/data), this code cell defines a function that computes a **train route between two station names** using a **line-aware Breadth-First Search (BFS)** on the railway graph.

---

## 1. Function Overview

### `train_route(src_name, dst_name, df_stations, graph, df_lines)`

The function:
- Takes a **source station name** and **destination station name**
- Searches the station graph
- Returns a **valid path of stations** that respects line constraints

---

## 2. Line Mapping

```python
line_map = df_lines.set_index("line_cd")["line_name"].to_dict()

In [13]:
def train_route(src_name, dst_name, df_stations, graph, df_lines):

    print(f"\n🚆 Finding route: {src_name} → {dst_name}")

    line_map = df_lines.set_index("line_cd")["line_name"].to_dict()
    
    src_rows = df_stations[df_stations["station_name"] == src_name]
    dst_rows = df_stations[df_stations["station_name"] == dst_name]

    if src_rows.empty or dst_rows.empty:
        return None

    # 🔥 MULTI-SOURCE BFS (all lines at source station)
    start_nodes = [(r["station_g_cd"], r["line_cd"]) for _, r in src_rows.iterrows()]
    end_stations = set(dst_rows["station_g_cd"])

    queue = collections.deque()
    visited = set()

    for node in start_nodes:
        queue.append((node, [node]))
        visited.add(node)

    while queue:

        (curr_station, curr_line), path = queue.popleft()

        # ✅ stop when we reach destination station (any line)
        if curr_station in end_stations:
            final_path = [s for (s, l) in path]
            print("✅ Route found:", final_path)
            return final_path

        for nei_station, nei_line in graph[(curr_station, curr_line)]:
        
            curr_line_name = line_map.get(curr_line)
            nei_line_name = line_map.get(nei_line)
        
            # 🔥 KEY RULE
            # stay on same line_name OR allow transfer at same station
            if not (
                nei_line_name == curr_line_name
                or nei_station == curr_station
            ):
                continue
        
            if (nei_station, nei_line) not in visited:
                visited.add((nei_station, nei_line))
                queue.append(
                    ((nei_station, nei_line), path + [(nei_station, nei_line)])
                )

    print("❌ No route found")
    return None

In [14]:
df_stations

,station_cd,station_g_cd,station_name,line_cd,post,address,lon,lat,open_ymd,close_ymd,e_status,e_sort
0,1110101,1110101,函館,11101,040-0063,北海道函館市若松町１２-１３,140.726413,41.773709,1902-12-10,0000-00-00,0,1110101
1,1110102,1110102,五稜郭,11101,041-0813,函館市亀田本町,140.733539,41.803557,0000-00-00,0000-00-00,0,1110102
2,1110103,1110103,桔梗,11101,041-0801,北海道函館市桔梗３丁目４１-３６,140.722952,41.846457,1902-12-10,0000-00-00,0,1110103
3,1110104,1110104,大中山,11101,041-1121,亀田郡七飯町大字大中山,140.713580,41.864641,0000-00-00,0000-00-00,0,1110104
4,1110105,1110105,七飯,11101,041-1111,亀田郡七飯町字本町,140.688556,41.886971,0000-00-00,0000-00-00,0,1110105
...,...,...,...,...,...,...,...,...,...,...,...,...
10931,9992719,9992719,てだこ浦西,99927,901-2102,沖縄県浦添市前田三丁目21,127.741861,26.241778,2019-10-01,0000-00-00,0,9992719
10932,9992801,1190202,九州鉄道記念館,99928,800-0000,福岡県北九州市門司区,130.962439,33.944392,0000-00-00,0000-00-00,0,9992801
10933,9992802,9992802,出光美術館,99928,800-0000,福岡県北九州市門司区,130.965292,33.947792,0000-00-00,0000-00-00,0,9992802
10934,9992803,9992803,ノーフォーク広場,99928,801-0854,福岡県北九州市門司区旧門司,130.964254,33.955973,0000-00-00,0000-00-00,0,9992803


### Explanation of the Code Cell

This code cell defines a function that **compresses a raw station path into readable journey segments grouped by train line**.

---

## 1. Function Overview

### `compress_route(df_stations, station_sequence, df_lines)`

The function:
- Takes a sequence of station IDs (from a computed route)
- Converts them into station names
- Groups consecutive stations by **line**
- Returns a simplified, human-readable route format

---

## 2. Line Mapping

```python
line_cd_to_name = df_lines.set_index("line_cd")["line_name"].to_dict()

In [15]:
def compress_route(df_stations, station_sequence, df_lines):

    line_cd_to_name = df_lines.set_index("line_cd")["line_name"].to_dict()

    segments = []

    current_line = None
    current_segment = []

    for station_cd in station_sequence:

        matches = df_stations[df_stations["station_g_cd"] == str(station_cd)]

        if matches.empty:
            continue

        row = matches.iloc[0]
        station = row["station_name"]
        line_name = line_cd_to_name.get(row["line_cd"], "Unknown")

        if current_line is None:
            current_line = line_name
            current_segment = [station]
            continue

        if line_name == current_line:
            if station not in current_segment:
                current_segment.append(station)
        else:
            segments.append((current_line, current_segment))
            current_line = line_name
            current_segment = [station]

    if current_segment:
        segments.append((current_line, current_segment))

    return segments

### Day Planning Optimisation (TSP) `solve_tsp()`

Each cluster = **1 day itinerary**

---

## Mathematical Model (MTZ Formulation)

### (a) Sets

- $N$: All locations (start + attractions)  
- $A \subseteq N$: Attractions  

---

### (b) Decision Variables

- $x_{ij} \in \{0,1\}$: travel from $i \to j$  
- $u_i$: MTZ variable  

---

### (c) Parameters

- $c_{ij}$: travel distance  
- $K = 3$: max attractions/day (Default)

---

### (d) Objective

$$
\min \sum_{i=1}^{N} \sum_{j=1}^{N} c_{ij} x_{ij}
$$

---

### (e) Constraints

#### Flow constraints:
$$
\sum_{i=1}^{N} x_{ij} = 1 \quad \forall j = 1, \dots, N
$$

$$
\sum_{j=1}^{N} x_{ij} = 1 \quad \forall i = 1, \dots, N
$$

#### MTZ subtour elimination:
$$
u_i - u_j + N x_{ij} \leq N - 1 \quad \forall i \ne j,\; 2 \leq i, j \leq N
$$

$$
1 \leq u_i \leq N - 1,\quad u_i \in \mathbb{Z} \quad \forall i = 2, \dots, N
$$

#### Binary:
$$
x_{ij} \in \{0,1\} \quad \forall i, j = 1, \dots, N
$$

$$
x_{ii} = 0 \quad \forall i = 1, \dots, N
$$

In [16]:
def solve_tsp(cluster_df, df_stations, start_lat, start_lon, K=3):
    print("\n🔹 Solving TSP...")

    # -------------------------
    # NODES
    # -------------------------
    attractions = [str(i) for i in cluster_df["id"]]
    nodes = ["START"] + attractions
    N = len(nodes)

    coords = {"START": (start_lat, start_lon)}
    for _, row in cluster_df.iterrows():
        coords[str(row["id"])] = (row["lat"], row["lon"])

    # -------------------------
    # MODEL
    # -------------------------
    m = Model("tsp_mtz")
    m.setParam("OutputFlag", 0)

    # x[i,j] = 1 if route goes from i to j
    x = m.addVars(nodes, nodes, vtype=GRB.BINARY, name="x")

    # MTZ variables only for attractions (exclude START)
    u = m.addVars(
        attractions,
        vtype=GRB.INTEGER,
        lb=1,
        ub=N - 1,
        name="u"
    )

    # -------------------------
    # COST MATRIX
    # -------------------------
    c = {
        (i, j): actual_distance(coords[i], coords[j])
        for i in nodes for j in nodes if i != j
    }

    # -------------------------
    # OBJECTIVE
    # -------------------------
    m.setObjective(
        quicksum(c[i, j] * x[i, j] for i in nodes for j in nodes if i != j),
        GRB.MINIMIZE
    )

    # -------------------------
    # CONSTRAINTS
    # -------------------------

    # No self-loops: x[i,i] = 0
    m.addConstrs((x[i, i] == 0 for i in nodes), name="no_self_loop")

    # Each node has exactly one outgoing arc
    m.addConstrs(
        (quicksum(x[i, j] for j in nodes) == 1 for i in nodes),
        name="out_flow"
    )

    # Each node has exactly one incoming arc
    m.addConstrs(
        (quicksum(x[i, j] for i in nodes) == 1 for j in nodes),
        name="in_flow"
    )

    # MTZ subtour elimination
    m.addConstrs(
        (
            u[i] - u[j] + N * x[i, j] <= N - 1
            for i in attractions for j in attractions if i != j
        ),
        name="mtz"
    )

    # Optional safeguard: enforce max attractions/day if needed
    if len(attractions) > K:
        print(f"⚠️ Warning: cluster has {len(attractions)} attractions, which exceeds K = {K}")

    # -------------------------
    # SOLVE
    # -------------------------
    m.optimize()

    if m.status != GRB.OPTIMAL:
        print("No optimal solution found.")
        return None

    # -------------------------
    # EXTRACT EDGES
    # -------------------------
    edges = [(i, j) for i in nodes for j in nodes if x[i, j].X > 0.5]

    # -------------------------
    # REBUILD ORDERED ROUTE
    # -------------------------
    route = ["START"]
    current = "START"

    while True:
        next_nodes = [j for (i, j) in edges if i == current]
        if not next_nodes:
            break

        nxt = next_nodes[0]
        route.append(nxt)

        if nxt == "START":
            break

        current = nxt

    print("Optimal route:", route)
    print("Objective value:", m.ObjVal)

    return route

# 🧭 Pre-Itinerary Construction Function — Explanation

This function builds a **step-by-step travel itinerary** between two locations by combining:

- 📍 Geographic proximity (nearest station search)  
- 🚆 Railway network routing (train graph traversal)  
- 🧹 Data cleaning & deduplication  
- 🗺️ Structured itinerary assembly  

---

## 1. Function Purpose

### `pre_itinerary(coord1, coord2, name1, name2, df_stations, graph)`

This function constructs a **real-world travel path** from:

- **Start location (coord1)** → nearest station → train network → nearest station → **end location (coord2)**

It outputs a structured DataFrame representing the full journey.

---

## 2. Initialize Itinerary Structure

```python
itinerary_rows = []

In [17]:
def pre_itinerary(coord1, coord2, name1, name2, df_stations, graph):

    print("\n🧭 Building pre-itinerary...")

    itinerary_rows = []

    # -------------------------
    # 1) START LOCATION
    # -------------------------
    itinerary_rows.append({
        "type": "location",
        "name": name1,
        "lat": coord1[0],
        "lon": coord1[1]
    })

    # -------------------------
    # 2) NEAREST STATIONS
    # -------------------------
    st1, _ = nearest_station(coord1, df_stations)
    st2, _ = nearest_station(coord2, df_stations)

    print(f"🚆 Routing: {st1['station_name']} → {st2['station_name']}")

    # add start station
    itinerary_rows.append({
        "type": "station",
        "name": st1["station_name"],
        "station_cd": st1["station_cd"],
        "station_g_cd": st1["station_g_cd"],
        "line_cd": st1["line_cd"],
        "lat": st1["lat"],
        "lon": st1["lon"]
    })

    # -------------------------
    # 3) TRAIN ROUTE (FULL PATH)
    # -------------------------
    route_path = train_route(
        st1["station_name"],
        st2["station_name"],
        df_stations,
        graph,
        df_lines
    )

    if route_path is None:
        route_path = []

    # -------------------------
    # 4) EXPAND ROUTE (FIXED VERSION)
    # -------------------------
    if len(route_path) > 0:

        cleaned_rows = []

        for g in route_path:

            # get all rows for this station group
            candidates = df_stations[df_stations["station_g_cd"] == g]

            # 🔥 pick ONE station only (prevents duplicates)
            row = candidates.sort_values("station_cd").iloc[0]

            cleaned_rows.append(row)

        route_df = pd.DataFrame(cleaned_rows)

        # remove endpoints (already added separately)
        route_df = route_df[
            ~route_df["station_cd"].isin([
                st1["station_cd"],
                st2["station_cd"]
            ])
        ]

        # final dedupe safety
        route_df = route_df.drop_duplicates(subset="station_cd")

        # append intermediate stations
        for _, row in route_df.iterrows():
            itinerary_rows.append({
                "type": "station",
                "name": row["station_name"],
                "station_cd": row["station_cd"],
                "station_g_cd": row["station_g_cd"],
                "line_cd": row["line_cd"],
                "lat": row["lat"],
                "lon": row["lon"]
            })

    else:
        print("⚠️ No train path found (fallback: direct)")

    # -------------------------
    # 5) END STATION
    # -------------------------
    itinerary_rows.append({
        "type": "station",
        "name": st2["station_name"],
        "station_cd": st2["station_cd"],
        "station_g_cd": st2["station_g_cd"],
        "line_cd": st2["line_cd"],
        "lat": st2["lat"],
        "lon": st2["lon"]
    })

    # -------------------------
    # 6) END LOCATION
    # -------------------------
    itinerary_rows.append({
        "type": "location",
        "name": name2,
        "lat": coord2[0],
        "lon": coord2[1]
    })

    # -------------------------
    # 7) FINAL CLEAN
    # -------------------------
    df_itinerary = pd.DataFrame(itinerary_rows)

    # remove consecutive duplicate stations
    df_itinerary = df_itinerary.loc[
        (df_itinerary["type"] != "station") |
        (df_itinerary["station_cd"] != df_itinerary["station_cd"].shift())
    ]

    print("\n✅ Pre-itinerary built:")
    print(df_itinerary)

    return df_itinerary

# 🧭 Day Itinerary Printing Function — Explanation

This function formats and prints a **human-readable travel itinerary** for a single day, converting raw structured route data into a **clean step-by-step travel plan**.

---

## 1. Function Purpose

### `print_day_itinerary(day_df, df_stations, df_lines, start_name="START")`

This function:
- Takes a structured itinerary DataFrame
- Removes redundancy
- Groups travel into logical segments
- Prints:
  - 📍 Locations
  - 🚇 Train lines
  - ➜ Station sequences

---

In [18]:
def print_day_itinerary(day_df, df_stations, df_lines, start_name="START"):

    print("\n🧭 ROUTE\n")

    # -------------------------
    # helper: replace START
    # -------------------------
    def display_name(name):
        return start_name if name == "START" else name

    # -------------------------
    # STEP 1: REMOVE CONSECUTIVE DUPLICATE LOCATIONS
    # -------------------------
    df = day_df.copy()

    df = df.loc[
        (df["type"] != "location") |
        (df["name"] != df["name"].shift())
    ].reset_index(drop=True)

    # -------------------------
    # STEP 2: SPLIT INTO BLOCKS
    # -------------------------
    blocks = []
    current_block = []

    for _, row in df.iterrows():

        current_block.append(row)

        # when we hit a location AFTER stations → block ends
        if row["type"] == "location" and len(current_block) > 1:
            blocks.append(pd.DataFrame(current_block))
            current_block = [row]

    if current_block:
        blocks.append(pd.DataFrame(current_block))

    # -------------------------
    # STEP 3: PRINT EACH BLOCK
    # -------------------------
    last_printed_location = None 
    
    for block in blocks:
    
        loc_rows = block[block["type"] == "location"]
    
        # -------------------------
        # START LOCATION
        # -------------------------
        start_loc = loc_rows.iloc[0]["name"]
    
        if start_loc != last_printed_location:
            print(f"\n📍 {display_name(start_loc)}")
            last_printed_location = start_loc
    
        # -------------------------
        # TRAIN SEGMENT
        # -------------------------
        station_seq = block[block["type"] == "station"]["station_g_cd"]
    
        if len(station_seq) > 0:
            station_seq = station_seq.astype(str).tolist()
    
            segments = compress_route(df_stations, station_seq, df_lines)
    
            for line, stations in segments:
                print(f"\n🚇 {line}")
                print("   " + " → ".join(stations))
    
        # -------------------------
        # END LOCATION
        # -------------------------
        if len(loc_rows) > 1:
            end_loc = loc_rows.iloc[-1]["name"]
    
            if end_loc != last_printed_location:
                print(f"\n📍 {display_name(end_loc)}")
                last_printed_location = end_loc

# 🚶 OSRM Walking Route Function — Explanation

This function generates a **real-world walking path between two coordinates** using the OSRM routing API, with a fallback to a straight-line approximation if the API fails.

---

## 1. Function Purpose

### `osrm_route(coord1, coord2)`

This function:
- Requests a **road-following walking route**
- Uses OpenStreetMap-based OSRM service
- Returns a **polyline of coordinates (lat, lon)**

If routing fails, it returns a **direct straight-line path**.

In [19]:
def osrm_route(coord1, coord2):
    """
    Returns road-following polyline between two coordinates using OSRM.
    """
    url = (
        f"http://router.project-osrm.org/route/v1/walking/"
        f"{coord1[1]},{coord1[0]};{coord2[1]},{coord2[0]}"
        f"?overview=full&geometries=geojson"
    )

    try:
        r = requests.get(url)
        data = r.json()
        coords = data["routes"][0]["geometry"]["coordinates"]
        # OSRM returns lon/lat → convert to lat/lon
        return [(lat, lon) for lon, lat in coords]
    except:
        # fallback straight line
        return [coord1, coord2]

# 🗺️ Day Map Visualisation Function — Explanation

This function generates an **interactive Folium map** that visualises a full day’s itinerary, including:

- 📍 Locations (start/end points & attractions)  
- 🚉 Train stations  
- 🚶 Walking routes (OSRM-based)  
- 🚆 Train connections  
- 🎛️ Layer controls & legend  

---

## 1. Function Purpose

### `plot_day_map(full_day, day_id="Day")`

This function:
- Takes a structured daily itinerary (`full_day`)
- Builds an interactive map
- Visualises movement between all points
- Differentiates **walking vs train travel**


In [20]:
def plot_day_map(full_day, day_id="Day"):

    center_lat = full_day["lat"].mean()
    center_lon = full_day["lon"].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

    # =========================
    # LAYERS
    # =========================
    start_layer = folium.FeatureGroup(name="Start Point").add_to(m)
    location_layer = folium.FeatureGroup(name="Locations").add_to(m)
    station_layer = folium.FeatureGroup(name="Stations").add_to(m)
    route_layer = folium.FeatureGroup(name="Routes").add_to(m)

    # =========================
    # NODE STYLES
    # =========================
    NODE_STYLE = {
        "start": {"color": "red", "icon": "play"},
        "location": {"color": "blue", "icon": "info-sign"},
        "station": {"color": "green", "icon": "train"}
    }

    # =========================
    # MARKERS
    # =========================
    for _, row in full_day.iterrows():

        node_type = row.get("type", "location")
        style = NODE_STYLE.get(node_type, NODE_STYLE["location"])

        marker = folium.Marker(
            location=[row["lat"], row["lon"]],
            popup=f"{node_type.upper()}: {row['name']}",
            icon=folium.Icon(color=style["color"], icon=style["icon"], prefix="fa")
        )

        if node_type == "start":
            marker.add_to(start_layer)
        elif node_type == "station":
            marker.add_to(station_layer)
        else:
            marker.add_to(location_layer)

    # =========================
    # SEGMENTED ROUTES
    # =========================
    rows = full_day.reset_index(drop=True)

    for i in range(len(rows) - 1):

        r1 = rows.iloc[i]
        r2 = rows.iloc[i + 1]

        coord1 = (r1["lat"], r1["lon"])
        coord2 = (r2["lat"], r2["lon"])

        type1 = r1["type"]
        type2 = r2["type"]

        # -------------------------
        # 🚶 WALKING SEGMENTS
        # -------------------------
        if (type1 != "station") or (type2 != "station"):

            route_coords = osrm_route(coord1, coord2)

            line = folium.PolyLine(
                route_coords,
                color="blue",
                weight=4,
                opacity=0.8,
                dash_array="5,10"   # dashed = walking
            ).add_to(route_layer)

            PolyLineTextPath(
                line,
                "➤",
                repeat=True,
                offset=6,
                attributes={"fill": "blue", "font-size": "12"}
            ).add_to(route_layer)

        # -------------------------
        # 🚆 TRAIN SEGMENTS
        # -------------------------
        else:

            route_coords = [
                [coord1[0], coord1[1]],
                [coord2[0], coord2[1]]
            ]

            line = folium.PolyLine(
                route_coords,
                color="red",
                weight=5,
                opacity=0.9
            ).add_to(route_layer)

            PolyLineTextPath(
                line,
                "➤",
                repeat=True,
                offset=6,
                attributes={"fill": "red", "font-size": "12"}
            ).add_to(route_layer)

    # =========================
    # LEGEND
    # =========================
    legend_html = f"""
    <div style="
        position: fixed;
        bottom: 50px;
        left: 50px;
        width: 180px;
        background-color: white;
        border:2px solid grey;
        z-index:9999;
        font-size:14px;
        padding:10px;
    ">
    <b>{day_id} Legend</b><br>
    🔴 Train Route<br>
    🔵 Walking Route<br>
    🚉 Station<br>
    📍 Location<br>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # =========================
    # LAYER CONTROL
    # =========================
    folium.LayerControl(collapsed=False).add_to(m)

    return m

In [21]:
def compute_total_distance(full_day):
    """
    Returns total distance travelled for one day.
    Priority:
    1) use an existing distance column if present
    2) otherwise compute from consecutive lat/lon rows
    """
    distance_cols = ["distance", "segment_distance", "travel_distance", "distance_km"]

    for col in distance_cols:
        if col in full_day.columns:
            return full_day[col].fillna(0).sum()

    total = 0.0
    df = full_day.reset_index(drop=True)

    if {"lat", "lon"}.issubset(df.columns):
        for i in range(len(df) - 1):
            coord1 = (df.loc[i, "lat"], df.loc[i, "lon"])
            coord2 = (df.loc[i + 1, "lat"], df.loc[i + 1, "lon"])
            total += actual_distance(coord1, coord2)

    return total

In [22]:
def plot_clusters_map(clusters, start_lat=None, start_lon=None, start_name="START"):
    if start_lat is not None and start_lon is not None:
        center_lat, center_lon = start_lat, start_lon
    else:
        first_cluster = clusters[0]
        center_lat = first_cluster.iloc[0]["lat"]
        center_lon = first_cluster.iloc[0]["lon"]

    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

    colors = [
        "red", "blue", "green", "purple", "orange",
        "darkred", "lightred", "beige", "darkblue", "darkgreen",
        "cadetblue", "darkpurple", "pink", "lightblue",
        "lightgreen", "gray", "black", "lightgray"
    ]

    if start_lat is not None and start_lon is not None:
        folium.Marker(
            [start_lat, start_lon],
            popup=start_name,
            tooltip=start_name,
            icon=folium.Icon(color="black", icon="home")
        ).add_to(m)

    for idx, cluster_df in enumerate(clusters, start=1):
        color = colors[(idx - 1) % len(colors)]
        fg = folium.FeatureGroup(name=f"Cluster {idx}")

        coords = []

        for _, row in cluster_df.iterrows():
            coords.append([row["lat"], row["lon"]])

            folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=6,
                popup=f"Cluster {idx}<br>{row['name']}",
                tooltip=f"Cluster {idx}: {row['name']}",
                color=color,
                fill=True,
                fill_opacity=0.9
            ).add_to(fg)

        if len(coords) >= 2:
            folium.PolyLine(coords, color=color, weight=3, opacity=0.8).add_to(fg)

        fg.add_to(m)

    folium.LayerControl().add_to(m)
    return m

# 🚀 Full Pipeline Orchestration Function — Explanation

This function is the **core driver of the entire travel planning system**, integrating:

- 📥 Data loading  
- 🧹 Preprocessing  
- 📍 Location clustering  
- 🧭 TSP optimisation (daily routing)  
- 🚆 Train + walking route construction  
- 🗺️ Map visualisation  
- 🧾 Human-readable itinerary output  

---

## 1. Function Purpose

### `run_model(...)`

This function:
- Executes the **end-to-end travel optimisation pipeline**
- Generates a **multi-day itinerary**
- Produces:
  - Optimised routes
  - Structured daily plans
  - Interactive maps
  - Printed travel guides


In [23]:
def run_model(df_lines, df_stations, df_joins, 
              locations_path, start_lat, start_lon,
              start_name="START", K=3):

    print("\n🚀 STARTING PIPELINE\n")

    df_locations = pd.read_csv(locations_path)
    df_locations["node_id"] = df_locations.index

    df_lines, df_stations, df_joins, df_locations = preprocess_data(
        df_lines, df_stations, df_joins, df_locations
    )

    # clusters = cluster_locations(df_locations, start_lat, start_lon, K)
    # clusters = greedy_spatial_cluster(df_locations, K)
    clusters = balanced_geo_cluster(df_locations, K)
    clusters = refine_clusters_by_swapping(clusters)
    print(clusters)
    
    # -----------------------------
    # 🗺️ CLUSTER MAP (NEW)
    # -----------------------------
    cluster_map = plot_clusters_map(
        clusters,
        start_lat=start_lat,
        start_lon=start_lon,
        start_name=start_name
    )
    display(cluster_map)

    graph = build_station_graph(df_stations, df_joins)

    full_itinerary = {}
    grand_total_distance = 0.0

    for day_idx, cluster_df in enumerate(clusters, start=1):

        print(f"\n================ DAY {day_idx} ================\n")

        tsp_result = solve_tsp(cluster_df, df_stations, start_lat, start_lon)

        if tsp_result is None:
            continue

        ordered_nodes = tsp_result
        ordered_nodes = [str(i) for i in ordered_nodes]

        node_map = cluster_df.set_index("id").astype(object).to_dict("index")
        node_map = {str(k): v for k, v in node_map.items()}

        node_map["START"] = {
            "name": start_name,
            "lat": start_lat,
            "lon": start_lon
        }

        ordered_locations = [
            {
                "id": i,
                "name": node_map[i]["name"],
                "lat": node_map[i]["lat"],
                "lon": node_map[i]["lon"]
            }
            for i in ordered_nodes
        ]

        day_plan = []

        for i in range(len(ordered_locations) - 1):

            loc1 = ordered_locations[i]
            loc2 = ordered_locations[i + 1]

            segment = pre_itinerary(
                (loc1["lat"], loc1["lon"]),
                (loc2["lat"], loc2["lon"]),
                loc1["name"],
                loc2["name"],
                df_stations,
                graph
            )

            day_plan.append(segment)

        full_day = pd.concat(day_plan, ignore_index=True)
        full_itinerary[f"Day_{day_idx}"] = full_day

        # -----------------------------
        # total distance per day
        # -----------------------------
        day_total_distance = compute_total_distance(full_day)
        grand_total_distance += day_total_distance
        print(f"📏 Total distance travelled for Day {day_idx}: {day_total_distance:.2f} km")

        # -----------------------------
        # 🗺️ MAP GENERATION PER DAY
        # -----------------------------
        day_map = plot_day_map(full_day, day_id=f"Day_{day_idx}")
        display(day_map)

        seen = set()
        rows = []

        for i, row in full_day.iterrows():
            if row["type"] == "location":

                # allow LAST row even if duplicate
                if i != len(full_day) - 1:
                    if row["name"] in seen:
                        continue
                    seen.add(row["name"])

            rows.append(row)

        full_day = pd.DataFrame(rows)

        print_day_itinerary(
            full_day,
            df_stations,
            df_lines,
            start_name=start_name
        )

    print(f"\n📦 Grand total distance travelled across all days: {grand_total_distance:.2f} km")
    return full_itinerary

In [24]:
results = run_model(
    df_lines=df_lines,
    df_stations=df_stations,
    df_joins=df_joins,
    locations_path= "selected_attractions.csv", # Input based on where selected_attractions.csv is saved.
    start_lat= 35.6568, # Input based on intended starting latitude.
    start_lon= 139.7459, # Input based on intended starting longitude.
    start_name="Tokyo Prince Hotel", # Input based on intended starting location name.
    K=3 # Input based on intended maximum number of attractions visited per day.
)


🚀 STARTING PIPELINE

🔧 Preprocessing data...
✅ Data cleaned
[   id                            name        lat         lon
0   4                     Shibuya Sky  35.658286  139.702262
1   6            Yebisu Brewery Tokyo  35.646988  139.708626
2   5  Shinjuku Gyoen National Garden  35.685072  139.709547,    id                                     name        lat         lon
0   0                     Ginza Central Street  35.669819  139.767258
1   3  Japan's Top 100 Streets (Ginza 3-chome)  35.672404  139.766779
2   2                 TOKYO Word Mark Monument  35.625316  139.775880,    id                                 name        lat         lon
0   8                             Sensō-ji  35.713403  139.795526
1   7                      Tokyo Dome City  35.705571  139.751970
2   1  Kawazu-Zakura next to Tokyo Skytree  35.709145  139.808684]



🔹 Building LINE-AWARE station graph...

================ DAY 1 ================


🔹 Solving TSP...
Restricted license - for non-production use only - expires 2027-11-29
Optimal route: ['START', '5', '4', '6', 'START']
Objective value: 12.518159447256114

🧭 Building pre-itinerary...
🚆 Routing: 赤羽橋 → 新宿御苑前

🚆 Finding route: 赤羽橋 → 新宿御苑前
✅ Route found: ['9930122', '2800916', '2800318', '2800116', '9930126', '1130207', '1130208', '1130208', '2800217', '2800216']

✅ Pre-itinerary built:
        type                            name        lat         lon  \
0   location              Tokyo Prince Hotel  35.656800  139.745900   
1    station                             赤羽橋  35.655007  139.743642   
2    station                            麻布十番  35.654682  139.737051   
3    station                             六本木  35.662836  139.731443   
4    station                           青山一丁目  35.672765  139.724159   
5    station                           国立競技場  35.679831  139.714932   
6    station    


🧭 ROUTE


📍 Tokyo Prince Hotel

🚇 都営大江戸線
   赤羽橋

🚇 東京メトロ南北線
   麻布十番

🚇 東京メトロ日比谷線
   六本木

🚇 東京メトロ銀座線
   青山一丁目

🚇 都営大江戸線
   国立競技場

🚇 JR山手線
   代々木 → 新宿

🚇 東京メトロ丸ノ内線
   新宿三丁目 → 新宿御苑前

📍 Shinjuku Gyoen National Garden

🚇 東京メトロ丸ノ内線
   新宿御苑前 → 新宿三丁目

🚇 JR山手線
   新宿 → 渋谷

📍 Shibuya Sky

🚇 JR山手線
   渋谷 → 恵比寿

📍 Yebisu Brewery Tokyo

🚇 JR山手線
   恵比寿

🚇 東京メトロ日比谷線
   広尾 → 六本木

🚇 東京メトロ南北線
   麻布十番

🚇 都営大江戸線
   赤羽橋

📍 Tokyo Prince Hotel

================ DAY 2 ================


🔹 Solving TSP...
Optimal route: ['START', '3', '0', '2', 'START']
Objective value: 12.28967863965095

🧭 Building pre-itinerary...
🚆 Routing: 赤羽橋 → 銀座一丁目

🚆 Finding route: 赤羽橋 → 銀座一丁目
✅ Route found: ['9930122', '9930121', '9930121', '1130102', '1130102', '2800111']

✅ Pre-itinerary built:
       type                                     name        lat         lon  \
0  location                       Tokyo Prince Hotel  35.656800  139.745900   
1   station                                      赤羽橋  35.655007  139.743642   
2   sta


🧭 ROUTE


📍 Tokyo Prince Hotel

🚇 都営大江戸線
   赤羽橋 → 大門

🚇 JR東海道本線(東京～熱海)
   新橋

🚇 東京メトロ銀座線
   銀座

📍 Japan's Top 100 Streets (Ginza 3-chome)

🚇 東京メトロ銀座線
   銀座

🚇 東京メトロ日比谷線
   東銀座

📍 Ginza Central Street

🚇 東京メトロ日比谷線
   東銀座

🚇 JR東海道本線(東京～熱海)
   新橋

🚇 JR山手線
   浜松町

🚇 東京モノレール
   天王洲アイル

🚇 りんかい線
   東京テレポート

📍 TOKYO Word Mark Monument

🚇 りんかい線
   東京テレポート

🚇 東京モノレール
   天王洲アイル

🚇 JR山手線
   浜松町

🚇 JR東海道本線(東京～熱海)
   新橋

🚇 都営大江戸線
   大門 → 赤羽橋

📍 Tokyo Prince Hotel

================ DAY 3 ================


🔹 Solving TSP...
Optimal route: ['START', '7', '8', '1', 'START']
Objective value: 18.88352740298672

🧭 Building pre-itinerary...
🚆 Routing: 赤羽橋 → 後楽園

🚆 Finding route: 赤羽橋 → 後楽園
✅ Route found: ['9930122', '2800916', '2800916', '2800915', '2800114', '2800115', '1131102', '1131315', '1131316', '2800204']

✅ Pre-itinerary built:
        type                name        lat         lon station_cd  \
0   location  Tokyo Prince Hotel  35.656800  139.745900        NaN   
1    station                 赤羽橋 


🧭 ROUTE


📍 Tokyo Prince Hotel

🚇 都営大江戸線
   赤羽橋

🚇 東京メトロ南北線
   麻布十番 → 六本木一丁目

🚇 東京メトロ銀座線
   溜池山王 → 赤坂見附

🚇 JR中央本線(東京～塩尻)
   四ツ谷

🚇 JR中央・総武線
   市ケ谷 → 飯田橋

🚇 東京メトロ丸ノ内線
   後楽園

📍 Tokyo Dome City

🚇 東京メトロ丸ノ内線
   後楽園 → 本郷三丁目

🚇 JR山手線
   御徒町

🚇 都営大江戸線
   新御徒町

🚇 つくばエクスプレス
   浅草

📍 Sensō-ji

🚇 つくばエクスプレス
   浅草

🚇 東武伊勢崎線
   浅草 → とうきょうスカイツリー

📍 Kawazu-Zakura next to Tokyo Skytree

🚇 東武伊勢崎線
   とうきょうスカイツリー → 押上〈スカイツリー前〉

🚇 JR中央・総武線
   錦糸町

🚇 東京メトロ半蔵門線
   清澄白河

🚇 東京メトロ東西線
   門前仲町

🚇 東京メトロ有楽町線
   月島

🚇 都営大江戸線
   勝どき → 築地市場 → 汐留 → 大門 → 赤羽橋

📍 Tokyo Prince Hotel

📦 Grand total distance travelled across all days: 69.46 km
